# ✈️ Notebook 1: Data Exploration and Project Lifecycle
**Project:** Flight Delay Prediction  
**Phase 1:** Traditional Machine Learning Only (Academic Syllabus)  
**Objective:** Predict whether a commercial flight arrives at least 15 minutes late (`Arrival Delay >= 15 min`) using only information available prior to scheduled departure.

---

## 📚 1. Machine Learning Project Lifecycle
In traditional machine learning, every rigorous predictive modeling project follows a structured seven-stage lifecycle:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                       MACHINE LEARNING PROJECT LIFECYCLE                    │
├─────────────────┬───────────────────────────────────────────────────────────┤
│ 1. Problem      │ Define the business and academic goal: Binary             │
│    Definition   │ classification to predict flight delay (>= 15 min)        │
│                 │ using only pre-departure attributes.                      │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 2. Data         │ Ingest flight performance datasets (e.g. US DOT BTS       │
│    Collection   │ or Kaggle). Audit schemas, file sizes, and availability.  │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 3. Exploratory  │ Inspect data types, missingness, duplicates, distributions│
│    Data Analysis│ class imbalance (~80% on-time, ~20% delayed), and delay   │
│    (EDA)        │ patterns across airlines, hubs, and hours.               │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 4. Data Pre-    │ Clean anomalies, exclude cancellations/diversions, create │
│    processing   │ chronological train/val/test splits, scale numericals,    │
│                 │ one-hot encode categoricals.                              │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 5. Model        │ Train Logistic Regression (MLE), compare Batch, Mini-Batch│
│    Training     │ and Stochastic Gradient Descent; tune L1/L2 penalties.    │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 6. Model        │ Evaluate Confusion Matrix, Precision, Recall, F1,         │
│    Evaluation   │ ROC-AUC, PR-AUC, and analyze the Bias-Variance tradeoff.  │
├─────────────────┼───────────────────────────────────────────────────────────┤
│ 7. Conclusion   │ Select optimal decision threshold, interpret feature      │
│    & Insights   │ coefficients, and summarize operational findings.         │
└─────────────────┴───────────────────────────────────────────────────────────┘
```

---

## ⚠️ 2. Strict Anti-Leakage Policy
- **Permitted Features (Pre-Departure Only):** Scheduled departure time, airline, origin, destination, scheduled flight duration, distance, calendar day/month.
- **Strictly Excluded (Data Leakage):** Actual departure time, departure delay, taxi-out, wheels-off, wheels-on, taxi-in, actual arrival time, delay reason breakdown codes.
- **Supervised Target:** Arrival delay is used **only** to compute the binary ground truth:
  $$y = \begin{cases} 1 & \text{if } \text{Arrival Delay} \ge 15 \text{ min} \\ 0 & \text{if } \text{Arrival Delay} < 15 \text{ min} \end{cases}$$


In [1]:
# Step 1: Import required libraries
import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path(os.path.abspath(".."))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    DATA_DIR,
    RAW_DATA_DIR,
    FIGURES_DIR,
    COLUMN_ALIASES,
    LEAKAGE_COLUMNS,
    TARGET_COLUMN,
    DELAY_THRESHOLD_MINUTES
)
from src.data_loader import (
    find_dataset_files,
    detect_column_mapping,
    audit_feature_leakage
)

# Configure plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["font.size"] = 11

print("✓ All libraries imported successfully.")
print(f"✓ Project Root: {PROJECT_ROOT}")


✓ All libraries imported successfully.
✓ Project Root: /Users/suryansh/Desktop/video/Flight_Delay_Prediction


In [2]:
# Step 2: Discover and inspect available CSV dataset files
dataset_files = find_dataset_files()

print("=" * 75)
print("WORKSPACE DATASET DISCOVERY")
print("=" * 75)

if not dataset_files:
    print("⚠️  DATASET STATUS: No CSV files currently detected in 'data/' or 'data/raw/'.")
    print("Please place your flight delay dataset (e.g. flights.csv from Kaggle 2015 BTS)")
    print(f"into: {RAW_DATA_DIR}/")
    DATASET_AVAILABLE = False
    raw_df = pd.DataFrame()
else:
    print(f"✓ Detected {len(dataset_files)} dataset file(s):")
    for idx, f in enumerate(dataset_files, 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  [{idx}] {f.name} ({size_mb:.2f} MB)")
    PRIMARY_FILE = dataset_files[0]
    DATASET_AVAILABLE = True
    
    # Read sample of 100k rows if file is very large (> 100MB) for fast exploration
    file_size_mb = PRIMARY_FILE.stat().st_size / (1024 * 1024)
    SAMPLE_ROWS = 100_000 if file_size_mb > 100 else None
    raw_df = pd.read_csv(PRIMARY_FILE, nrows=SAMPLE_ROWS, low_memory=False)
    print(f"✓ Successfully loaded: {len(raw_df):,} rows, {len(raw_df.columns)} columns.")


WORKSPACE DATASET DISCOVERY
⚠️  DATASET STATUS: No CSV files currently detected in 'data/' or 'data/raw/'.
Please place your flight delay dataset (e.g. flights.csv from Kaggle 2015 BTS)
into: /Users/suryansh/Desktop/video/Flight_Delay_Prediction/data/raw/


In [3]:
# Step 3: Column Inspection and Canonical Concept Mapping
if DATASET_AVAILABLE and not raw_df.empty:
    mapping, missing_concepts = detect_column_mapping(raw_df.columns)
    
    print("CANONICAL CONCEPT MAPPING:")
    print("-" * 65)
    concepts_to_check = [
        ("Scheduled Departure", "scheduled_departure"),
        ("Scheduled Arrival", "scheduled_arrival"),
        ("Airline / Carrier", "airline"),
        ("Origin Airport", "origin"),
        ("Destination Airport", "destination"),
        ("Distance", "distance"),
        ("Arrival Delay (Target Source)", "arrival_delay"),
        ("Cancellations", "cancellation"),
        ("Diversions", "diversion"),
    ]
    for label, concept in concepts_to_check:
        found = mapping.get(concept, "❌ MISSING")
        print(f"  {label:<32} -> {found}")
    print("-" * 65)
else:
    print("Awaiting dataset file to perform actual column mapping.")


Awaiting dataset file to perform actual column mapping.


In [4]:
# Step 4: Data Quality - Rows, Types, Missing Values, Duplicates, and Date Range
if DATASET_AVAILABLE and not raw_df.empty:
    print(f"Total Rows       : {len(raw_df):,}")
    print(f"Total Columns    : {len(raw_df.columns)}")
    print(f"Duplicate Rows   : {raw_df.duplicated().sum():,}")
    
    # Missing values
    missing = raw_df.isnull().sum()
    missing_pct = (missing / len(raw_df)) * 100
    missing_df = pd.DataFrame({"Missing Count": missing, "Missing Pct (%)": missing_pct})
    print("\nTop Columns by Missing Percentage:")
    print(missing_df.sort_values(by="Missing Pct (%)", ascending=False).head(10))
else:
    print("Awaiting dataset file to report exact missing values and row statistics.")


Awaiting dataset file to report exact missing values and row statistics.


In [5]:
# Step 5: Anti-Leakage Audit
if DATASET_AVAILABLE and not raw_df.empty:
    leakage_cols, safe_cols = audit_feature_leakage(raw_df.columns)
    print("=" * 70)
    print("ANTI-LEAKAGE FEATURE SEGREGATION")
    print("=" * 70)
    print(f"✓ Safe Pre-Departure Features : {len(safe_cols)}")
    print(f"⚠️ Banned Post-Departure Columns : {len(leakage_cols)}")
    for col in leakage_cols:
        print(f"  - ❌ [EXCLUDE FROM MODEL] {col}")
    print("=" * 70)
else:
    print("Awaiting dataset file to audit leakage columns.")


Awaiting dataset file to audit leakage columns.


## 📝 Notebook 1 Summary & Next Steps
- **Project Lifecycle:** Detailed the 7 core stages from Problem Definition to Evaluation and Conclusion.
- **Data Integrity:** Inspected dataset presence, schema aliases, missing values, and duplicate rows without fabricating data.
- **Anti-Leakage Guardrail:** Post-departure columns are identified and strictly barred from feature sets.
- **Next Notebook:** `02_data_preprocessing.ipynb` will implement operational filtering, chronological train/val/test splitting, and feature transformations.
